In [1]:
#@title 1 · Settings & GPU check
BRANCH    = "main"   #@param {type:"string"}
USE_DRIVE = False    #@param {type:"boolean"}
HF_TOKEN  = ""       #@param {type:"string"}
PORT = 3900

import os, sys, secrets, subprocess, time, re, urllib.request, urllib.error
REPO_DIR = "/content/VoiceStudio"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/VoiceStudio_data"
else:
    DATA_DIR = "/content/omnivoice_data"
os.makedirs(DATA_DIR, exist_ok=True)

key_file = f"{DATA_DIR}/.api_key"
if os.path.exists(key_file):
    API_KEY = open(key_file).read().strip()
else:
    API_KEY = secrets.token_urlsafe(32)
    open(key_file, "w").write(API_KEY)

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: Runtime → Change runtime type → T4 GPU"

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
#@title 2 · System packages + clone repo
!apt-get -qq update > /dev/null && apt-get -qq install -y ffmpeg libsndfile1 > /dev/null
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b {BRANCH} https://github.com/debpalash/VoiceStudio.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Already up to date.
/content/VoiceStudio


In [3]:
#@title 3 · Build the web UI (~3–6 min)
!curl -fsSL https://bun.sh/install | bash -s "bun-v1.4.2" > /dev/null 2>&1
os.environ["PATH"] = os.path.expanduser("~/.bun/bin") + ":" + os.environ["PATH"]
os.environ["ELECTRON_SKIP_BINARY_DOWNLOAD"] = "1"   # headless, no Electron binary needed
!bun --version
!bun install --frozen-lockfile || bun install
!bun run --cwd electron build:web
!test -f frontend/dist/index.html && echo "✅ UI built" || echo "❌ frontend/dist missing, check output above"

1.4.2
bun install v1.4.2 (744846f84)
  📦 Installing [2/1168] 
Checked 1168 installs across 1324 packages (no changes) [1040.00ms]
$ bun run locale:check && vite build --config vite.web.config.ts
$ node tests/locale-encoding.mjs && node tests/locale-source-keys.mjs && node tests/locale-coverage.mjs
Locale encoding and interpolation: 21 files passed
Locale source keys: 1662 static renderer keys exist in en.json
{
  "missingEntries": 0,
  "byNamespace": {},
  "orphans": {}
}
vite v8.3.0 building client environment for production...
✓ 8789 modules transformed.
computing gzip size...
../frontend/dist/early-error-capture.js                                                        0.94 kB
../frontend/dist/assets/paoloantinori-DOzNqVNY.jpg                                             4.53 kB
../frontend/dist/assets/marreiradigital-BvjQOH1m.jpg                                           4.89 kB
../frontend/dist/assets/kevin9327-fJ-qtGUy.png                                                 5.46 kB
..

In [4]:
#@title 4 · Backend dependencies (~5–10 min)
!pip -q install uv
!uv pip install --system --constraint deploy/torch-constraints.txt .

cudnn_target = os.path.join(sys.prefix, "lib", "python%d.%d" % sys.version_info[:2], "site-packages", "cudnn8_compat")
!pip -q install --no-deps --target {cudnn_target} nvidia-cudnn-cu12==8.9.7.29

!python3 -c "import torch; print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())"

Using Python 3.13.15 environment at: /usr
Resolved 237 packages in 805ms
Prepared 1 package in 488ms
Uninstalled 1 package in 8ms
Installed 1 package in 2ms
 ~ omnivoice==0.5.7 (from file:///content/VoiceStudio)
torch 2.8.0+cu128 | CUDA: True


In [5]:
#@title 5 · Start VoiceStudio backend
if "backend" in globals() and backend.poll() is None:
    backend.terminate(); time.sleep(3)

env = os.environ.copy()
env.update({
    "PYTHONPATH": f"{REPO_DIR}/backend",
    "PYTHONUNBUFFERED": "1",
    "OMNIVOICE_SERVER_MODE": "1",
    "OMNIVOICE_API_KEY": API_KEY,
    "OMNIVOICE_DATA_DIR": DATA_DIR,
    "HF_HOME": f"{DATA_DIR}/huggingface",
    "OMNIVOICE_PORT": str(PORT),
})
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN

backend = subprocess.Popen(
    ["python3", "-m", "uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=REPO_DIR, env=env,
    stdout=open("/content/backend.log", "w"), stderr=subprocess.STDOUT,
)

def get(path):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=5) as r:
            return r.status, r.read().decode()
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode()
    except Exception:
        return None, ""

for i in range(120):

    if backend.poll() is not None:
        print("Backend exited:\n", open("/content/backend.log").read()[-3000:]); break
    code, body = get("/health")

    if code == 200:
        print("\nBackend healthy:", body); break

    _, prog = get("/startup/progress")
    print(f"[{i*5}s] starting… {prog[:100]}", end="\r")
    time.sleep(5)


✅ Backend healthy: {"status":"ok","device":"cuda (Tesla T4)","version":"0.5.7"}


In [6]:
#@title 6 · Public Cloudflare link
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

if "tunnel" in globals() and tunnel.poll() is None:
    tunnel.terminate()

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT,
)

public_url = None

for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        public_url = m.group(0); break

print("VoiceStudio:", public_url or "not ready, see /content/cloudflared.log")
print("Admin key (paste when the UI asks):", API_KEY)

🌐 VoiceStudio: https://breaking-dear-supports-envelope.trycloudflare.com
🔑 Admin key (paste when the UI asks): D7KcS5yO5syXj6yrdxhClr3XTlNh9rTt6yRDCYo3rWc


In [7]:
#@title 7 · Backend logs
!tail -n 40 /content/backend.log

INFO:     Started server process [25205]
INFO:     Waiting for application startup.
2026-10-12 10:36:46,812 INFO [omnivoice.api] Startup watchdog armed: thread dump if startup exceeds 300s (#632).
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:3900 (Press CTRL+C to quit)
2026-10-12 10:36:47,433 INFO [omnivoice.api] Published 2 media-tool directories on PATH so dependencies can find ffmpeg/ffprobe (#1256)
INFO:     127.0.0.1:48450 - "GET /health HTTP/1.1" 503 Service Unavailable
INFO:     127.0.0.1:48456 - "GET /startup/progress HTTP/1.1" 200 OK
2026-10-12 10:36:51,774 INFO [omnivoice.hf_progress] hf_progress: installed tqdm patch on huggingface_hub.utils.tqdm
2026-10-12 10:36:52,396 INFO [omnivoice.setup.models] Loaded 38 models from /content/VoiceStudio/backend/config/models.yaml
2026-10-12 10:36:52,777 INFO [omnivoice.model] downloads: Xet disabled → legacy LFS (hf_xet 1.6.0 installed=True), high_perf=False
2026-10-12 10:36:54,659 INFO [omnivoic

In [ ]:
while True:
    time.sleep(60)